In [2]:
import requests, hashlib, time, os, json
from pathlib import Path
from datetime import datetime
from math import log10
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# load_dotenv()
# API_KEY    = os.getenv('PODCASTINDEX_API_KEY')
# API_SECRET = os.getenv('PODCASTINDEX_API_SECRET')
# if not API_KEY or not API_SECRET:
#     raise ValueError('Add PODCASTINDEX_API_KEY and PODCASTINDEX_API_SECRET to your .env file.')

# BASE_URL   = 'https://api.podcastindex.org/api/1.0'
OUTPUT_DIR = Path('../data')
OUTPUT_DIR.mkdir(exist_ok=True)

This is the function clean description. I removed fuzzy matching and adding HTML tag removal

In [3]:
import re
import string
import spacy

nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
DEFAULT_STOPWORDS = nlp.Defaults.stop_words | {"podcast", "episode"}

def clean_description(text, stopwords=DEFAULT_STOPWORDS, do_lower=True):
    """
    Preprocess text by:
    - Removing HTML tags
    - Lowercasing (optional)
    - Removing punctuation
    - Lemmatization
    - Removing stopwords
    """
    if not isinstance(text, str):
        return ''

    # Drop HTML tags like <p>, </div>, etc.
    text = re.sub(r'<[^>]+>', ' ', text)

    if do_lower:
        text = text.lower()

    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    doc = nlp(text)
    words = [token.lemma_ for token in doc if token.is_alpha and token.text not in stopwords]

    result = ' '.join(words)
    result = re.sub(r'\s+', ' ', result).strip()
    return result

In [ ]:
INPUT_PATH = OUTPUT_DIR / 'podcasts_cleaned2.csv'
OUTPUT_PATH =  OUTPUT_DIR / 'podcasts_cleanedHTML2.csv'
CHUNKSIZE = 50

first = True
for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
    chunk['clean_description'] = chunk['description'].apply(clean_description)
    chunk.to_csv(OUTPUT_PATH, mode='w' if first else 'a', index=False, header=first)
    first = False
    print(f"Processed {len(chunk)} rows...")

print(f"Saved cleaned podcast data to {OUTPUT_PATH}")


split epsidoes csv into two csvs

Hannah runs this code cell below

In [ ]:
INPUT_PATH = OUTPUT_DIR / 'episodes_cleaned2.csv'
OUTPUT_PATH1 =  OUTPUT_DIR / 'episodes_cleanedHTML1.csv'
OUTPUT_PATH2 =  OUTPUT_DIR / 'episodes_cleanedHTML2.csv'
CHUNKSIZE = 50
first = True
for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
    chunk['clean_description'] = chunk['description'].apply(clean_description)
    chunk.to_csv(OUTPUT_PATH1 if first else OUTPUT_PATH2, mode='w', index=False, header=first)
    first = False
    print(f"Processed {len(chunk)} rows...")
    
print(f"Saved cleaned episode data to {OUTPUT_PATH1}")

Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 rows...
Processed 50 

Other person runs this: ESTELLA run this one

In [ ]:
INPUT_PATH = OUTPUT_DIR / 'episodes_cleaned2.csv'
OUTPUT_PATH1 =  OUTPUT_DIR / 'episodes_cleanedpart1.csv'
OUTPUT_PATH2 =  OUTPUT_DIR / 'episodes_cleanedpart2.csv'
first = False
for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNKSIZE):
    chunk['clean_description'] = chunk['description'].apply(clean_description)
    chunk.to_csv(OUTPUT_PATH2, mode='w' if first else 'a', index=False, header=first)
    first = False
    print(f"Processed {len(chunk)} episode rows...")

print(f"Saved cleaned episode data to {OUTPUT_PATH2}")

In [ ]:
#Combine back into one csv called episodes_cleaned2HTML.csv
df1 = pd.read_csv(OUTPUT_PATH1)
df2 = pd.read_csv(OUTPUT_PATH2)
combined_df = pd.concat([df1, df2], ignore_index=True)
combined_df.to_csv(OUTPUT_DIR / 'episodes_cleanedHTML2.csv', index=False)
print(f"Combined cleaned episode data saved to {OUTPUT_DIR / 'episodes_cleaned2HTML.csv'}")

Open the csv files and check that most tags are removed beore continuing please please please

In [ ]:
# Note: Mjght skip
# --- Podcast Embedding: 0.7*episode centroid + 0.3*description embedding ---
import numpy as np
from collections import defaultdict
from sklearn.preprocessing import normalize

# Load episode and podcast description embeddings
show_embs = np.load(OUTPUT_DIR / 'embeddings/description_embeddings_improved2.npy')  # shape: (n_podcasts, emb_dim)
ep_embs = np.load(OUTPUT_DIR / 'episode_embeddings_improved2.npy')        # shape: (n_episodes, emb_dim)
show_ids = [line.strip() for line in (OUTPUT_DIR / 'embeddings/embedding_show_ids2.txt').read_text().splitlines()]
ep_ids = [line.strip() for line in (OUTPUT_DIR / 'embeddings/embedding_episode_ids2.txt').read_text().splitlines()]

# Save the old podcast description embeddings before overwriting
# np.save(OUTPUT_DIR / 'old_description_embeddings.npy', show_embs.astype(np.float32))

# Map podcast and episode IDs to their indices
show_id_list = show_ids  # already str
show_id_to_idx = {sid: i for i, sid in enumerate(show_id_list)}
ep_id_list = ep_ids      # already str

df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes_cleanedHTML2.csv')

# Group episode indices by podcast_id
podcast_to_ep_indices = defaultdict(list)
for i, row in enumerate(df_episodes.itertuples(index=False)):
    podcast_to_ep_indices[str(row.podcast_id)].append(i)

# Compute new podcast embeddings
new_show_embs = np.zeros_like(show_embs)
for sid, show_idx in show_id_to_idx.items():
    ep_indices = podcast_to_ep_indices.get(sid, [])
    if ep_indices:
        centroid = ep_embs[ep_indices].mean(axis=0)
        centroid = normalize(centroid.reshape(1, -1), norm='l2').flatten()  # L2 normalize the centroid
        new_show_embs[show_idx] = 0.1 * centroid + 0.9 * show_embs[show_idx]
    else:
        new_show_embs[show_idx] = show_embs[show_idx]  # fallback: just description embedding

np.save(OUTPUT_DIR / 'description_embeddings_mixed2.npy', new_show_embs.astype(np.float32))
print('Saved old podcast embeddings to old_description_embeddings.npy')
print('Saved new podcast embeddings (0.1*episode centroid + 0.9*description)')


Get embeddings now with the cleaned html csv  files. First for podcasts

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

df_podcasts = pd.read_csv(OUTPUT_DIR / 'podcasts_cleanedHTML2.csv')

# Build text strings: for podcasts, combine name + description + categories; for episodes, combine episode name + description.
def podcast_text(r):
    cats = str(r.get('categories', '')).replace('|', ' ')
    return f"{r['name']} {cats} {r.get('description', '')}"

def ep_text(r):
    return f"{r['episode_name']} {r.get('description', '')}"

show_texts = df_podcasts.apply(podcast_text, axis=1).tolist()
ep_texts   = df_episodes.apply(ep_text,     axis=1).tolist()
show_ids   = df_podcasts['id'].astype(str).tolist()
ep_ids     = df_episodes['id'].astype(str).tolist()

# Podcast SVD embeddings
N_COMPONENTS = 100  # number of SVD dimensions 100-300 for now cuz um thats what i learned

tfidf_shows = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix_shows = tfidf_shows.fit_transform(show_texts) # sparse (n_podcasts, 10000)
print(f'Podcast TF-IDF matrix: {tfidf_matrix_shows.shape}')

svd_shows = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
show_embs = svd_shows.fit_transform(tfidf_matrix_shows)  # dense (n_podcasts, 100)
print(f'Explained variance: {svd_shows.explained_variance_ratio_.sum():.2%}')

np.save(OUTPUT_DIR / 'description_embeddings_improved3.npy', show_embs.astype(np.float32))
Path(OUTPUT_DIR / 'embedding_show_ids3.txt').write_text('\n'.join(show_ids))
print(f'Podcast embeddings: shape={show_embs.shape} | {(OUTPUT_DIR/"description_embeddings_improved3.npy").stat().st_size/1024:.0f} KB')

Create episode embeddings

In [ ]:
# Episode SVD embeddings:
def episode_embeddings(file_num):
    df_episodes = pd.read_csv(OUTPUT_DIR / f'episodes_cleanedHTML{file_num}.csv')
    tfidf_eps = TfidfVectorizer(max_features=10000, stop_words='english')
    tfidf_matrix_eps = tfidf_eps.fit_transform(ep_texts)
    print(f'Episode TF-IDF matrix: {tfidf_matrix_eps.shape}')

    svd_eps = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
    ep_embs = svd_eps.fit_transform(tfidf_matrix_eps)

    np.save(OUTPUT_DIR / f'episode_embeddings_improved{file_num}.npy', ep_embs.astype(np.float32))
    Path(OUTPUT_DIR / f'embedding_episode_ids{file_num}.txt').write_text('\n'.join(ep_ids))
    print(f'Episode embeddings:  shape={ep_embs.shape} | {(OUTPUT_DIR/f"episode_embeddings_improved{file_num}.npy").stat().st_size/1024:.0f} KB')

    # Save vectorizers + SVD models (needed at query time!) 
    import pickle
    with open(OUTPUT_DIR / f'svd_shows_improved{file_num}.pkl', 'wb') as f:
        pickle.dump({'tfidf': tfidf_shows, 'svd': svd_shows}, f)
    with open(OUTPUT_DIR / f'svd_episodes_improved{file_num}.pkl', 'wb') as f:
        pickle.dump({'tfidf': tfidf_eps, 'svd': svd_eps}, f)
    print(f'Saved svd_shows_improved{file_num}.pkl and svd_episodes_improved{file_num}.pkl')
    

episode_embeddings(2)

Now get the mixed embeddings

In [ ]:
# Note: Mjght skip
# --- Podcast Embedding: 0.7*episode centroid + 0.3*description embedding ---
import numpy as np
from collections import defaultdict
from sklearn.preprocessing import normalize

# Load episode and podcast description embeddings
show_embs = np.load(OUTPUT_DIR / 'embeddings/description_embeddings_improved3.npy')  # shape: (n_podcasts, emb_dim)
ep_embs = np.load(OUTPUT_DIR / 'episode_embeddings_improved3.npy')        # shape: (n_episodes, emb_dim)
show_ids = [line.strip() for line in (OUTPUT_DIR / 'embeddings/embedding_show_ids3.txt').read_text().splitlines()]
ep_ids = [line.strip() for line in (OUTPUT_DIR / 'embeddings/embedding_episode_ids3.txt').read_text().splitlines()]

# Save the old podcast description embeddings before overwriting
# np.save(OUTPUT_DIR / 'old_description_embeddings.npy', show_embs.astype(np.float32))

# Map podcast and episode IDs to their indices
show_id_list = show_ids  # already str
show_id_to_idx = {sid: i for i, sid in enumerate(show_id_list)}
ep_id_list = ep_ids      # already str

df_episodes = pd.read_csv(OUTPUT_DIR / 'episodes_cleaned3.csv')

# Group episode indices by podcast_id
podcast_to_ep_indices = defaultdict(list)
for i, row in enumerate(df_episodes.itertuples(index=False)):
    podcast_to_ep_indices[str(row.podcast_id)].append(i)

# Compute new podcast embeddings
new_show_embs = np.zeros_like(show_embs)
for sid, show_idx in show_id_to_idx.items():
    ep_indices = podcast_to_ep_indices.get(sid, [])
    if ep_indices:
        centroid = ep_embs[ep_indices].mean(axis=0)
        centroid = normalize(centroid.reshape(1, -1), norm='l2').flatten()  # L2 normalize the centroid
        new_show_embs[show_idx] = 0.1 * centroid + 0.9 * show_embs[show_idx]
    else:
        new_show_embs[show_idx] = show_embs[show_idx]  # fallback: just description embedding

np.save(OUTPUT_DIR / 'description_embeddings_mixed3.npy', new_show_embs.astype(np.float32))
# print('Saved old podcast embeddings to old_description_embeddings.npy')
print('Saved new podcast embeddings (0.1*episode centroid + 0.9*description)')
